# Target disease normalizer

- integrating diseases, drugs and targets from the TTD
- Using the DTNPD for extra bolstering and disease info 

In [ ]:
#/////////////////////////////////////////// input files
targets_file = ''
drugs_file = ''
target_drug = ''
target_disease = ''
drug_disease = ''
dntpd_targets = ''
dntpd_diseases = ''
dntpd_drugs = ''
diseases_with_materialised_paths_file = ''
#/////////////////////////////////////////// output files
targets_out_file = ''
drugs_out_file = ''
diseases_out_file = ''
target_disease_out = ''
target_drug_out = ''
#/////////////////////////////////////////// imports
import pandas as pd
import re
#//////////////////////////////////////////// extract targets and diseases
#functions
def add_approved_neuro(diseases):
    icd_term = r"\[ICD-11:\s*([68][A-Z0-9]*(?:\.[A-Z0-9]+)?(?:\s*-\s*[A-Z0-9]+(?:\.[A-Z0-9]+)?)?)\s*\]"
    icds = []
    for disease in diseases:
        if disease.startswith('Approved'):
            if re.search(icd_term, disease):
                pat = re.compile(icd_term)
                icd = pat.findall(disease)
                icds.extend(icd)
    if icds:
        return list(dict.fromkeys(icds))
    else:
        return pd.NA
#read csv
print(f'Reading {target_disease}...', flush=True)
with open(target_disease) as f:
    target_disease_df = pd.DataFrame(columns=['ttd_target_id','field_name','field'])
    for line_num, line in enumerate(f, start=1):
        if line_num < 23:
            continue
        if line.strip():
            line = line.strip()
            line = line.split(maxsplit=2)
            target_disease_df.loc[len(target_disease_df)] = line 
# process csv
print('Processing target disease file...', flush=True)
target_disease_df = target_disease_df.loc[~target_disease_df['field_name'].eq('TARGNAME')].copy()
target_disease_df = target_disease_df.loc[~target_disease_df['field_name'].eq('TARGETID')].copy()

target_disease_df =target_disease_df.drop(columns=['field_name'])

target_disease_df = target_disease_df.groupby('ttd_target_id').agg(add_approved_neuro)
target_disease_df = target_disease_df.loc[target_disease_df['field'].notna()].copy()
target_disease = target_disease.rename(columns = {'field': 'icd_11'})
#///////////////////////////////////////// extracting targets
print(f'Reading {targets_file} and extracting only neuro targets...',flush=True)
neuro_targets = set(target_disease['ttd_target_id'].to_list())
rows =[]
with open(targets_file) as f:
    for line_num, line in enumerate(f, start=1):
        if line_num > 33:
            if line.strip():
                line = line.strip()
                line = line.split(maxsplit=2)
                if line[0] in neuro_targets:
                    rows.append(line)
                else:
                    continue
# removing unnecessary columns and grouping all metadata to id and unstacking
print('Unpacking targets...',flush=True)
target_df = pd.DataFrame(rows,columns=['ttd_target_id','field_name','field'])
target_df = target_df.loc[~target_df['field_name'].isin(['TARGETID','FORMERID','DRUGINFO','SEQUENCE'])].copy()
target_df = target_df.groupby(['ttd_target_id','field_name'])['field'].agg(list).unstack('field_name').reset_index()
#////////////////////////////////////////// extracting disease drug 
# functions
def get_icd(value):
    if pd.isna(value):
        return pd.NA
    icd_term = r"ICD-11:\s*([68][A-Z0-9]*(?:\.[A-Z0-9]+)?(?:\s*-\s*[A-Z0-9]+(?:\.[A-Z0-9]+)?)?)\s*"
    icds = []
    if not re.search(icd_term, value):
        return pd.NA
    values = value.split('|')
    for value in values:
        phase = ''
        codes = []
        if re.search(icd_term, value):
            pat = re.compile(icd_term)
            codes = pat.findall(value)
            if 'Approved in' in value:
                vals = value.split()
                phase = ' '.join(vals[-3:])
            elif 'Phase' in value:
                vals = value.split()
                phase = ' '.join(vals[-2:])
            elif 'Approved' in value:
                vals = value.split()
                phase = ' '.join(vals[-1:])
            for code in codes:
                icds.append(f'{code},{phase}')
    return icds
# getting drug ID name and pahse for icd 6/8
print('Extracting neuro disease drug pairs from {drug_disease}...',flush=True)
with open(drug_disease) as f:
    drug_disease_df = pd.DataFrame(columns = ['ttd_id','name','indication'])
    use = []
    d_id = None
    d_name = None
    for line_num, line in enumerate(f,start=1):
        if line_num > 22:
            line = line.strip()
            if not line: 
                continue
            line = line.split(maxsplit=1) if line else line
            
            if line[0] == 'TTDDRUID':
                if use and d_id is not None:
                    entry = [d_id,d_name,'|'.join(use)]
                    drug_disease_df.loc[len(drug_disease_df)] = entry
                d_id = line[1]
                d_name = None
                use = []
            elif line[0] == 'DRUGNAME':
                d_name = line[1]
            elif line[0] == 'INDICATI':
                use.append(line[1])
    if use and d_id is not None:
        entry = [d_id, d_name, '|'.join(use)]
        drug_disease_df.loc[len(drug_disease_df)] = entry

drug_disease_df['icds'] = drug_disease_df.apply(lambda row: get_icd(row.indication),axis = 1)
drug_disease_df
drug_disease_df = drug_disease_df.explode('icds', ignore_index=True).dropna(subset=['icds'])
if drug_disease_df.empty:
    drug_disease_df['icd'] = pd.Series(dtype='string')
    drug_disease_df['phase'] = pd.Series(dtype='string')
else:
    drug_disease_df[['icd', 'phase']] = drug_disease_df['icds'].str.split(',', n=1, expand=True)

drug_disease_df = drug_disease_df.drop(columns=['icds','indication'])

#///////////////////////////////////////// extracting drugs
print('Extracting drugs with neuro from {drugs_file}...')
neuro_list = set(drug_disease_df['ttd_id'].to_list())
with open('/home/school/masters/Scripts/Theraputic_Target_Database/P1-02-TTD_drug_download.txt') as f:
    drugs_df = pd.DataFrame(columns=['ttd_id','field_name','field'])
    for line_num, line in enumerate(f,start=1):
        if line_num > 28:
            if line.strip():
                line = line.split(maxsplit=2)
                ttid = line[0]
                if ttid not in neuro_list:
                    continue
                field_name = line[1]
                field = line[2]
                drugs_df.loc[len(drugs_df)] = [ttid,field_name,field]
print('Unstcking and grouping by id')
drugs_df = drugs_df.loc[~drugs_df['field_name'].isin(['DRUGCOMP', 'DRUGTYPE'])].copy()
drugs_df = drugs_df.groupby(['ttd_id','field_name'])['field'].agg(list).unstack('field_name').reset_index()

#//////////////////////////////////////////// using icd-11 materialized paths from WHO 
print('Loading materliaised paths for DB {diseases_with_materialised_paths_file}', flush = True)
all_icd_11 = pd.read_csv(diseases_with_materialised_paths_file)
print('Making diseases from WHO into wand formats', flush = True)
all_icd_11['id'] = range(1,len(all_icd_11)+1)
#//////////////////////////////////////////// matching the icd-11 codes from WHO so the other drug disease and  can have the wand ID
def get_icd_wand_ids(icd_list):
    wand_ids = []
    for icds in icd_list:
        if '-' in icds[0]:
            icd_start = icds.split('-')[0]
            icd_end = icds.split('-')[1]
            insert more code here
        else:
            disease_id = all_icd_11[all_icd_11['code']==icds[0], 'id'][0]
            wand_ids.append(disease_id)
    return '|'.join(wand_ids)

target_disease_df['wand_ids'] = target_disease_df.apply(lambda row: get_icd_wand_ids(row.field))
target_disease_df['wand_ids'] = target_disease_df.str.split('|',regex=False)
target_disease_df = target_disease_df.explode('wand_ids', ignore_index=True)
# so for some reason for the drugs I exploded already but did not for the targets above so the method has to be slightly different here